# 05 - Production Model Evaluation, Calibration, KS Statistic & SHAP Explainability

This notebook conducts comprehensive model validation on the held-out test partition, evaluates probability calibration, computes Kolmogorov-Smirnov (KS) separation, and generates SHAP feature attribution explanations.

In [ ]:
import sys, os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve

sys.path.append('..')
from src.data_loader import load_raw_data, get_stratified_split
from src.metrics import calculate_credit_metrics, pd_to_credit_score, assign_risk_tier, calculate_ks_statistic
from src.explain import CreditRiskExplainer

plt.style.use('seaborn-v0_8-whitegrid')

# 1. Load Production Pipeline and Metadata
pipeline = joblib.load('../artifacts/credit_scoring_pipeline_v1.joblib')
with open('../artifacts/model_metadata.json', 'r') as f:
    metadata = json.load(f)

threshold = float(metadata['optimal_threshold'])
print(f"Loaded production pipeline version {metadata['model_version']}. Operating threshold: {threshold:.4f}")

## 1. Evaluate Holdout Test Partition

Inference is executed on the isolated 15% test partition.

In [ ]:
df = load_raw_data()
_, _, _, _, X_test, y_test = get_stratified_split(df)

y_test_prob = pipeline.predict_proba(X_test)[:, 1]
test_metrics = calculate_credit_metrics(y_test, y_test_prob, threshold=threshold)

print("=== Holdout Test Set Performance ===")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

## 2. Kolmogorov-Smirnov (KS) Separation & ROC Curves

In [ ]:
fpr, tpr, thresholds_roc = roc_curve(y_test, y_test_prob)
ks_stat, ks_thresh = calculate_ks_statistic(y_test, y_test_prob)

plt.figure(figsize=(12, 5))

# ROC Curve
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color='navy', label=f'ROC-AUC = {test_metrics["roc_auc"]:.4f}')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()

# KS Plot
plt.subplot(1, 2, 2)
plt.plot(thresholds_roc, tpr, label='Cumulative Defaults (Bad)', color='red')
plt.plot(thresholds_roc, fpr, label='Cumulative Non-Defaults (Good)', color='green')
plt.axvline(x=ks_thresh, color='black', linestyle=':', label=f'KS = {ks_stat:.4f} (at {ks_thresh:.3f})')
plt.xlim([0, 0.4])
plt.xlabel('Cutoff Probability')
plt.ylabel('Cumulative Distribution')
plt.title('Kolmogorov-Smirnov (KS) Separation')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Probability Calibration (Reliability Diagram)

Verifying that predicted probabilities correspond to empirical default rates.

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, y_test_prob, n_bins=10)

plt.figure(figsize=(6, 6))
plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label='Calibrated XGBoost')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Empirical Default Fraction')
plt.title(f'Calibration Reliability Diagram (Brier Score: {test_metrics["brier_score"]:.4f})')
plt.legend()
plt.show()

## 4. SHAP Explainability & Adverse Action Reason Codes

In [ ]:
explainer = CreditRiskExplainer(pipeline)
sample_applicant = X_test.iloc[[0]]

explanation = explainer.explain_instance(sample_applicant)
print("=== Applicant Adverse Action Reasons ===")
for idx, reason in enumerate(explanation['adverse_action_reasons'], 1):
    print(f"{idx}. {reason}")